In [1]:
import pyodbc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from imblearn.over_sampling import SMOTE

In [2]:
conn=pyodbc.connect("Driver={SQL Server};"
                    "Server=.\SQLEXPRESS;"
                    "Database=Bank_Analysis;"
                    "Trusted_conn=yes;")

In [3]:
connection=conn.cursor()

In [4]:
client_df=pd.read_sql("Select * from client",conn)
account_df=pd.read_sql("select * from account",conn)
disposition_df=pd.read_sql("Select * from disp",conn)
district_df=pd.read_sql("Select * from district",conn)
order_df=pd.read_sql("Select * from [order]",conn)
card_df=pd.read_sql("Select * from [card]",conn)
loan_df=pd.read_sql("Select * from loan",conn)
transaction_df=pd.read_sql("select * from master_transaction",conn)

In [5]:
loan_model = loan_df.merge(
    account_df,
    on='account_id',
    how='left'
)

In [6]:
loan_model["loan_risk"] = loan_df["status"].map({
    "Running Contract": 0,
    "Contract Finished": 0,
    "Client in Debt": 1,
    "Loan Not Paid": 1
})

In [7]:
loan_model["loan_risk"].value_counts()

loan_risk
0.0    606
1.0     31
Name: count, dtype: int64

In [8]:
owner = disposition_df[disposition_df['type']=='OWNER'][
    ['account_id','client_id']
]

loan_model = loan_model.merge(
    owner,
    on='account_id',
    how='left'
)

In [9]:
loan_model = loan_model.merge(
    client_df[['client_id','age','sex']],
    on='client_id',
    how='left'
)

In [10]:
loan_model = loan_model.merge(
    district_df,
    on='district_id',
    how='left'
)

In [11]:
txn_features = transaction_df.groupby('account_id').agg(

    total_transactions=('trans_id','count'),

    total_transaction_amount=('amount','sum'),

    avg_transaction_amount=('amount','mean'),

    avg_balance=('balance','mean'),

    max_balance=('balance','max'),

    min_balance=('balance','min')

).reset_index()

In [12]:
loan_model = loan_model.merge(
    txn_features,
    on='account_id',
    how='left'
)

In [13]:
order_features = order_df.groupby('account_id').agg(

    total_orders=('order_id','count'),

    avg_order_amount=('amount','mean')

).reset_index()

In [14]:
loan_model = loan_model.merge(
    order_features,
    on='account_id',
    how='left'
)

In [15]:
card_features = disposition_df.merge(
    card_df,
    on='disp_id',
    how='inner'
)

card_features = card_features.groupby(
    'account_id'
).agg(

    total_cards=('card_id','count')

).reset_index()

In [16]:
loan_model = loan_model.merge(
    card_features,
    on='account_id',
    how='left'
)

In [17]:
loan_model.fillna(0, inplace=True)

,loan_id,account_id,date_x,amount,duration,payments,status,district_id,frequency,date_y,...,crime_statistics2,total_transactions,total_transaction_amount,avg_transaction_amount,avg_balance,max_balance,min_balance,total_orders,avg_order_amount,total_cards
0,4959,2,2017-01-05,80952,24,3373.0,Contract Finished,1,Monthly Issuance,2018-02-26,...,99107,477,3151357,6606.618449,36527.942497,69302.000000,1100.000000,2,5319.349976,0.0
1,4961,19,2019-04-29,30276,12,2523.0,Loan Not Paid,21,Monthly Issuance,2020-04-07,...,2354,301,1575463,5234.096346,16698.486457,58157.500000,-5817.600098,1,2523.199951,0.0
2,4962,25,2020-12-08,30276,12,2523.0,Contract Finished,68,Monthly Issuance,2021-07-28,...,5887,272,2955058,10864.183824,56471.617112,134209.906250,900.000000,4,2653.549988,0.0
3,4967,37,2021-10-14,318480,60,5308.0,Client in Deb,20,Monthly Issuance,2022-08-18,...,1542,128,943257,7369.195312,37417.641136,104761.500000,-1011.200012,4,2576.375000,0.0
4,4968,38,2021-04-19,110736,48,2307.0,Running Contract,19,Weekly Issuanace,2022-08-08,...,1099,129,571761,4432.255814,34383.792204,55991.101562,13841.000000,4,2416.700012,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677,7294,11327,2021-09-27,39168,24,1632.0,Running Contract,7,Monthly Issuance,2022-10-15,...,4846,74,559590,7562.027027,56410.387748,81495.601562,300.000000,2,2662.500000,0.0
678,7295,11328,2021-07-18,280440,60,4674.0,Running Contract,54,Monthly Issuance,2021-11-05,...,18696,144,1312188,9112.416667,46713.342346,136153.000000,1000.000000,1,4674.000000,0.0
679,7304,11349,2018-10-29,419880,60,6998.0,Running Contract,1,Weekly Issuanace,2020-05-26,...,99107,302,3953292,13090.370861,48868.020194,113678.703125,200.000000,2,8942.000000,0.0
680,7305,11359,2019-08-06,54024,12,4502.0,Contract Finished,61,Monthly Issuance,2019-10-01,...,2059,376,2943914,7829.558511,35898.573406,81705.796875,1000.000000,3,3091.433268,1.0


In [18]:
loan_model = pd.get_dummies(
    loan_model,
    columns=['sex'],
    drop_first=True
)

In [19]:
features = [

'amount',

'duration',

'payments',

'age',

'total_transactions',

'total_transaction_amount',

'avg_transaction_amount',

'avg_balance',

'max_balance',

'min_balance',

'total_orders',

'avg_order_amount',

'total_cards',

'population',

'urban_population_ratio',

'average_salary',

'unemployment_rates2',

'entrepreneurs_per_1000_resident',

'crime_statistics2',

'sex_Male'

]

In [20]:
loan_model.columns

Index(['loan_id', 'account_id', 'date_x', 'amount', 'duration', 'payments',
       'status', 'district_id', 'frequency', 'date_y', 'Account_type',
       'card_assigned', 'loan_risk', 'client_id', 'age', 'district_name',
       'region', 'population', 'municipality_statistics1',
       'municipality_statistics2', 'municipality_statistics3',
       'municipality_statistics4', 'number_of_cities',
       'urban_population_ratio', 'average_salary', 'unemployment_rates2',
       'entrepreneurs_per_1000_resident', 'crime_statistics2',
       'total_transactions', 'total_transaction_amount',
       'avg_transaction_amount', 'avg_balance', 'max_balance', 'min_balance',
       'total_orders', 'avg_order_amount', 'total_cards', 'sex_Male'],
      dtype='str')

In [21]:
X = loan_model[features]

y = loan_model['loan_risk']

In [22]:
X_train,X_test,y_train,y_test = train_test_split(

X,

y,

test_size=0.20,

random_state=42,

stratify=y

)

In [23]:
model=LogisticRegression().fit(X_train,y_train)

In [24]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)


In [25]:
metrics.confusion_matrix(y_train,train_pred)

array([[516,   4],
       [ 14,  11]])

In [26]:
print(metrics.classification_report(y_train,train_pred))

              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98       520
         1.0       0.73      0.44      0.55        25

    accuracy                           0.97       545
   macro avg       0.85      0.72      0.77       545
weighted avg       0.96      0.97      0.96       545



In [27]:
smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [28]:
model1=LogisticRegression().fit(X_train_smote,y_train_smote)

In [29]:
train_pred1 = model1.predict(X_train_smote)
test_pred1 = model1.predict(X_test)


In [30]:
metrics.confusion_matrix(y_train_smote,train_pred1)

array([[486,  34],
       [ 16, 504]])

In [31]:
print(metrics.classification_report(y_train_smote,train_pred1))

              precision    recall  f1-score   support

         0.0       0.97      0.93      0.95       520
         1.0       0.94      0.97      0.95       520

    accuracy                           0.95      1040
   macro avg       0.95      0.95      0.95      1040
weighted avg       0.95      0.95      0.95      1040



In [32]:
metrics.confusion_matrix(y_test,test_pred1)

array([[124,   7],
       [  1,   5]])

In [33]:
print(metrics.classification_report(y_test,test_pred1))

              precision    recall  f1-score   support

         0.0       0.99      0.95      0.97       131
         1.0       0.42      0.83      0.56         6

    accuracy                           0.94       137
   macro avg       0.70      0.89      0.76       137
weighted avg       0.97      0.94      0.95       137



In [34]:
import pickle


# features save
pickle.dump(
    features,
    open("features.pkl", "wb")
)

In [35]:
import joblib

joblib.dump(model, "loan_risk_model.joblib")

['loan_risk_model.joblib']